# Chapter 5 Companion — Homework (Favre-Averaged, Compressible RANS)

These problems use the canonical dataset generated in the companion notebook:

- **Data:** `Chapter5Companion_Favre_Canonical.csv`
- **Variance/flux data:** `Chapter5Companion_Favre_VarianceProfiles.csv` (will be rebuilt if missing)

Work in the provided cells; keep units on all plots and reported values.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
plt.rcParams.update({'figure.dpi': 120})

csv_canon = Path('Chapter5Companion_Favre_Canonical.csv')
csv_var   = Path('Chapter5Companion_Favre_VarianceProfiles.csv')

def load_or_build_variance():
    if csv_var.exists():
        return pd.read_csv(csv_var)
    # build if missing using definitions consistent with the companion notebook
    dfc = pd.read_csv(csv_canon)
    y    = dfc['y_m'].values
    rho  = dfc['rho_kgm3'].values
    lm   = dfc['lm_m'].values
    nu_t = dfc['nu_t_m2s'].values
    k    = np.clip(dfc['k_m2s2'].values, 1e-12, None)
    dTdy = dfc['dTdy_Km'].values
    dYdy = dfc['dYdy_1m'].values
    Pr_t, Sc_t = 0.9, 0.7
    C_theta, C_Y = 0.7, 0.7
    kappa_t = nu_t/Pr_t
    D_t     = nu_t/Sc_t
    Jyt = - rho * D_t * dYdy
    P_var_T = kappa_t * (dTdy**2)
    vartheta_T_eq = P_var_T * lm / (C_theta * np.sqrt(k))
    chi_T_eq = C_theta * np.sqrt(k) * vartheta_T_eq / np.clip(lm, 1e-12, None)
    P_var_Y = D_t * (dYdy**2)
    vartheta_Y_eq = P_var_Y * lm / (C_Y * np.sqrt(k))
    chi_Y_eq = C_Y * np.sqrt(k) * vartheta_Y_eq / np.clip(lm, 1e-12, None)
    out = pd.DataFrame({
        'y_m': y,
        'kappa_t_m2s': kappa_t,
        'D_t_m2s': D_t,
        'J_y_t_kgm2s': Jyt,
        'P_var_T_K2s': P_var_T,
        'vartheta_T_eq_K2': vartheta_T_eq,
        'chi_T_eq_K2s': chi_T_eq,
        'P_var_Y_1s': P_var_Y,
        'vartheta_Y_eq_-': vartheta_Y_eq,
        'chi_Y_eq_1s': chi_Y_eq,
    })
    out.to_csv(csv_var, index=False)
    return out


## Problem 1 — Reproduce and label the mean profiles
Load `Chapter5Companion_Favre_Canonical.csv`, and plot:
(a) $U(y)$ [m/s], (b) $T(y)$ [K], (c) $\rho(y)$ [kg/m$^3$], (d) $\mu_t(y)$ [Pa·s].
Add publication-style titles and axis labels.

In [ ]:
df = pd.read_csv('Chapter5Companion_Favre_Canonical.csv')
y = df['y_m'].values
for col, ylab, title in [
    ('U_ms','U [m/s]','Mean Velocity'),
    ('T_K','T [K]','Mean Temperature'),
    ('rho_kgm3','rho [kg/m^3]','Mean Density'),
    ('mu_t_Pas','mu_t [Pa s]','Eddy Viscosity'),
]:
    fig, ax = plt.subplots(); ax.plot(y, df[col])
    ax.set_xlabel('y [m]'); ax.set_ylabel(ylab); ax.set_title(title)
    plt.show()

## Problem 2 — TKE terms and residual check (steady, 1D)
Compute and plot the modeled terms of the TKE balance:
$P_s=\mu_t (\partial_y U)^2$, $\bar\rho\,\varepsilon$, and the diffusion term
$d/dy[(\mu/\sigma_k + \mu_t/\sigma_k^t)\,dk/dy]$ (use $\mu=1.8\times10^{-5}$ Pa·s and $\sigma_k=\sigma_k^t=1$).
Compute the residual $R(y)=P_s-\bar\rho\,\varepsilon+\text{diffusion}$. Report $\min R$, $\max R$ and discuss briefly.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, pandas as pd
df = pd.read_csv('Chapter5Companion_Favre_Canonical.csv')
y = df['y_m'].values
mu_t = df['mu_t_Pas'].values
U = df['U_ms'].values
k = df['k_m2s2'].values
rho = df['rho_kgm3'].values
eps = df['eps_m2s3'].values
dUdy = np.gradient(U, y)
dkdy = np.gradient(k, y)
mu = 1.8e-5
sigma_k = sigma_k_t = 1.0
Ps = mu_t * dUdy**2
reps = rho * eps
coef = mu/sigma_k + mu_t/sigma_k_t
diffusion = np.gradient(coef*dkdy, y)
R = Ps - reps + diffusion
print('min R, max R =', R.min(), R.max())
fig, ax = plt.subplots(); ax.plot(y, Ps, label='P_s'); ax.plot(y, reps, label='rho*eps'); ax.plot(y, diffusion, label='diffusion')
ax.set_xlabel('y [m]'); ax.set_ylabel('[SI]'); ax.set_title('TKE Terms'); ax.legend(frameon=False); plt.show()
fig, ax = plt.subplots(); ax.plot(y, R)
ax.set_xlabel('y [m]'); ax.set_ylabel('Residual [SI]'); ax.set_title('Modeled TKE Residual'); plt.show()

## Problem 3 — Turbulent mass flux & diffusivities
(a) Compute $J_y^{(t)}= -\rho D_t\,dY/dy$ with $D_t=\nu_t/\mathrm{Sc}_t$ and $\mathrm{Sc}_t=0.7$.
(b) Plot $J_y^{(t)}(y)$ and $\kappa_t(y)=\nu_t/\mathrm{Pr}_t$ with $\mathrm{Pr}_t=0.9$.
(c) Briefly comment on where flux magnitudes are largest and why.

In [ ]:
df = pd.read_csv('Chapter5Companion_Favre_Canonical.csv')
y    = df['y_m'].values
rho  = df['rho_kgm3'].values
nu_t = df['nu_t_m2s'].values
dYdy = df['dYdy_1m'].values
Pr_t, Sc_t = 0.9, 0.7
D_t = nu_t/Sc_t
kappa_t = nu_t/Pr_t
Jyt = - rho * D_t * dYdy
fig, ax = plt.subplots(); ax.plot(y, Jyt)
ax.set_xlabel('y [m]'); ax.set_ylabel('J_y^t [kg/(m^2 s)]'); ax.set_title('Turbulent Mass Flux (Species)'); plt.show()
fig, ax = plt.subplots(); ax.plot(y, kappa_t)
ax.set_xlabel('y [m]'); ax.set_ylabel('kappa_t [m^2/s]'); ax.set_title('Turbulent Thermal Diffusivity'); plt.show()

## Problem 4 — Temperature variance budget (modeled)
Using `Chapter5Companion_Favre_VarianceProfiles.csv` (or rebuild via `load_or_build_variance()`),
plot the temperature-variance **production** $\mathcal{P}_{\vartheta_T}$ and modeled **dissipation** $\chi_T^{(eq)}$.
Report the integral $\int_0^1 \mathcal{P}_{\vartheta_T}\,dy$ and $\int_0^1 \chi_T^{(eq)}\,dy$ and comment on their comparison.

In [ ]:
dfv = load_or_build_variance()
y = dfv['y_m'].values
P_T = dfv['P_var_T_K2s'].values
chi_T = dfv['chi_T_eq_K2s'].values
fig, ax = plt.subplots(); ax.plot(y, P_T, label='Production'); ax.plot(y, chi_T, label='Dissipation (eq)')
ax.set_xlabel('y [m]'); ax.set_ylabel('[K^2/s]'); ax.set_title('Temperature Variance Budget'); ax.legend(frameon=False); plt.show()
Ip = np.trapz(P_T, y); Id = np.trapz(chi_T, y)
print('Integrals: int P_T dy =', Ip, ',  int chi_T dy =', Id)

## Problem 5 — Species variance budget (modeled)
Repeat Problem 4 for species variance (use $\mathcal{P}_{\vartheta_Y}$ and $\chi_Y^{(eq)}$).
Comment on similarities/differences relative to the temperature case.

In [ ]:
dfv = load_or_build_variance()
y = dfv['y_m'].values
P_Y = dfv['P_var_Y_1s'].values
chi_Y = dfv['chi_Y_eq_1s'].values
fig, ax = plt.subplots(); ax.plot(y, P_Y, label='Production'); ax.plot(y, chi_Y, label='Dissipation (eq)')
ax.set_xlabel('y [m]'); ax.set_ylabel('[1/s]'); ax.set_title('Species Variance Budget'); ax.legend(frameon=False); plt.show()
Ip = np.trapz(P_Y, y); Id = np.trapz(chi_Y, y)
print('Integrals: int P_Y dy =', Ip, ',  int chi_Y dy =', Id)

## Problem 6 — Sensitivity to closure constants
(a) Vary $\mathrm{Pr}_t\in\{0.7, 0.9, 1.2\}$; re-compute and overplot $\kappa_t(y)$.
(b) Vary $\mathrm{Sc}_t\in\{0.5, 0.7, 1.0\}$; re-compute and overplot $J_y^{(t)}(y)$.
(c) Vary $C_\theta, C_Y\in\{0.5, 0.7, 1.0\}$ and assess how equilibrium variances change.

In [ ]:
dfc = pd.read_csv('Chapter5Companion_Favre_Canonical.csv')
y    = dfc['y_m'].values
rho  = dfc['rho_kgm3'].values
nu_t = dfc['nu_t_m2s'].values
dTdy = dfc['dTdy_Km'].values
dYdy = dfc['dYdy_1m'].values
k    = np.clip(dfc['k_m2s2'].values, 1e-12, None)
lm   = dfc['lm_m'].values

# (a) Pr_t sensitivity on kappa_t
fig, ax = plt.subplots()
for Pr_t in [0.7, 0.9, 1.2]:
    kappa_t = nu_t/Pr_t
    ax.plot(y, kappa_t, label=f'Pr_t={Pr_t}')
ax.set_xlabel('y [m]'); ax.set_ylabel('kappa_t [m^2/s]'); ax.set_title('Pr_t Sensitivity'); ax.legend(frameon=False); plt.show()

# (b) Sc_t sensitivity on J_y^t
fig, ax = plt.subplots()
for Sc_t in [0.5, 0.7, 1.0]:
    D_t = nu_t/Sc_t
    Jyt = - rho * D_t * dYdy
    ax.plot(y, Jyt, label=f'Sc_t={Sc_t}')
ax.set_xlabel('y [m]'); ax.set_ylabel('J_y^t [kg/(m^2 s)]'); ax.set_title('Sc_t Sensitivity'); ax.legend(frameon=False); plt.show()

# (c) C_theta, C_Y sensitivity on equilibrium variances
fig, ax = plt.subplots()
for C_theta in [0.5, 0.7, 1.0]:
    kappa_t = nu_t/0.9
    vartheta_T_eq = (kappa_t*(dTdy**2))*lm/(C_theta*np.sqrt(k))
    ax.plot(y, vartheta_T_eq, label=f'C_theta={C_theta}')
ax.set_xlabel('y [m]'); ax.set_ylabel('vartheta_T^(eq) [K^2]'); ax.set_title('C_theta Sensitivity'); ax.legend(frameon=False); plt.show()

fig, ax = plt.subplots()
for C_Y in [0.5, 0.7, 1.0]:
    D_t = nu_t/0.7
    vartheta_Y_eq = (D_t*(dYdy**2))*lm/(C_Y*np.sqrt(k))
    ax.plot(y, vartheta_Y_eq, label=f'C_Y={C_Y}')
ax.set_xlabel('y [m]'); ax.set_ylabel('vartheta_Y^(eq) [-]'); ax.set_title('C_Y Sensitivity'); ax.legend(frameon=False); plt.show()


## Problem 7 — Add buoyancy (optional extension)
Assume a constant $N^2$ and evaluate $P_b=-(\mu_t/\mathrm{Pr}_t)N^2$. Overplot $P_s$ and $P_b$ and discuss where buoyancy is comparable to shear production.
Try $N^2\in\{\pm 5\times10^{-4}\ \text{s}^{-2}\}$.

In [ ]:
df = pd.read_csv('Chapter5Companion_Favre_Canonical.csv')
y = df['y_m'].values; mu_t = df['mu_t_Pas'].values
U = df['U_ms'].values; rho = df['rho_kgm3'].values
dUdy = np.gradient(U, y)
Ps = mu_t * dUdy**2
Pr_t = 0.9
for N2 in [5e-4, -5e-4]:
    Pb = -(mu_t/Pr_t)*N2
    fig, ax = plt.subplots(); ax.plot(y, Ps, label='P_s'); ax.plot(y, Pb, label=f'P_b (N^2={N2:+.1e})')
    ax.set_xlabel('y [m]'); ax.set_ylabel('[SI]'); ax.set_title('Shear vs Buoyancy Production'); ax.legend(frameon=False); plt.show()
